In [1]:
import pandas as pd

df = pd.read_csv("housing.csv")

print(df.isna().mean() * 100)
print(df.info())

longitude             0.000000
latitude              0.000000
housing_median_age    0.000000
total_rooms           0.000000
total_bedrooms        1.002907
population            0.000000
households            0.000000
median_income         0.000000
median_house_value    0.000000
ocean_proximity       0.000000
dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="median_house_value")
y = df["median_house_value"]

# Il parametro stratify di Scikit-Learn è 
# progettato per problemi di classificazione, dove il target y assume valori discreti

X_train, X_temp, y_train, y_temp = train_test_split(X,y, test_size=0.20, random_state=42)

val_size = 0.05 / 0.20

print(val_size)

X_test, X_val, y_test, y_val = train_test_split(X_temp,y_temp, test_size=val_size, random_state=42)


0.25


In [3]:
# FEATURE ENGINEERING - Prima del preprocessing
# Le variabili multicollineari sono:"total_rooms","total_bedrooms","population",

# 1. Calcolo dei rapporti e rimozione delle feature collineari
datasets = [X_train, X_test, X_val]

for dataset in datasets:
    # Creazione delle metriche relative
    dataset["total_rooms_per_households"] = (
        dataset["total_rooms"] / dataset["households"]
    )
    dataset["population_per_households"] = (
        dataset["population"] / dataset["households"]
    )

    # Rimozione delle grandezze collineari, MANTENENDO households come riferimento di scala
    dataset.drop(
        columns=[
            "total_rooms",
            "total_bedrooms",
            "population",
        ],
        inplace=True,
    )

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_features = X_train.select_dtypes(include="number").columns
cat_features = X_train.select_dtypes(include=["object", "bool", "string"]).columns

num_pipeline = Pipeline(steps=[
    # non sono presenti features numeriche da imputare dopo l'eliminazione di total_bedrooms
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    # non sono presenti features categoriche da imputare 
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

preprocessor.set_output(transform="pandas")

X_train_processed = preprocessor.fit_transform(X_train,y_train)
X_test_processed = preprocessor.transform(X_test)
X_val_processed = preprocessor.transform(X_val)

In [5]:
# Analisi multicollinearita' 
import numpy as np
corr = X_train_processed.corr()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)) # prendiamo solo la matrice triangolare superiore
print(upper[abs(upper) > 0.80].stack())

num__longitude  num__latitude   -0.924485
dtype: float64


Le features combinate sono state aggiunte precedentemente poichè dopo la scalature avrebbero perso informazioni, essendo scalate non avrebbe avuto senso effettuarne un rapporto


In [22]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

lasso = Lasso(random_state=42, max_iter=3000)
selector = SelectFromModel(estimator=lasso).fit(X_train_processed, y_train)

selector.set_output(transform='pandas')

X_train_selected = selector.transform(X_train_processed)
X_test_selected = selector.transform(X_test_processed)
X_val_selected = selector.transform(X_val_processed)

print(f"Features selezionate:{selector.get_feature_names_out()}")



Features selezionate:['num__longitude' 'num__latitude' 'num__housing_median_age'
 'num__households' 'num__median_income' 'num__total_rooms_per_households'
 'num__population_per_households' 'cat__ocean_proximity_<1H OCEAN'
 'cat__ocean_proximity_INLAND' 'cat__ocean_proximity_ISLAND'
 'cat__ocean_proximity_NEAR OCEAN']


In [ ]:

# Regressione Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

param_grid_ridge = {
    'alpha' : [1,2,3],
    'max_iter' : [200,500,1000]
}

ridge = Ridge(random_state=42, solver='auto')
cv = KFold(n_splits=5, random_state=42, shuffle=True)

grid_ridge = GridSearchCV(estimator=ridge, param_grid=param_grid_ridge, cv=cv, scoring='neg_mean_squared_error')
grid_ridge.fit(X_train_selected, y_train)

best_params = grid_ridge.best_params_
best_estimator = grid_ridge.best_estimator_


ridge_pred_test = best_estimator.predict(X_test_selected)
ridge_pred_train = best_estimator.predict(X_train_selected)

mse_ridge_test = mean_squared_error(y_test, ridge_pred_test)
mse_ridge_train = mean_squared_error(y_train, ridge_pred_train)

print(f"MSE RIDGE TRAIN: {mse_ridge_train:.2f}")
print(f"MSE RIDGE TEST: {mse_ridge_test:.2f}")


r2_ridge_test = r2_score(y_test, ridge_pred_test)
r2_ridge_train = r2_score(y_train, ridge_pred_train)

print(f"R2 RIDGE TRAIN: {r2_ridge_train:.2f}")
print(f"R2 RIDGE TEST: {r2_ridge_test:.2f}")

print(f"Best Params: {best_params}")
print(f"Best Estimator: {best_estimator}")



MSE RIDGE TRAIN: 5092741553.61
MSE RIDGE TEST: 5358639925.58
R2 RIDGE TRAIN: 0.62
R2 RIDGE TEST: 0.60
Best Params: {'alpha': 1, 'max_iter': 200}
Best Estimator: Ridge(alpha=1, max_iter=200, random_state=42)


In [31]:
from sklearn.ensemble import RandomForestRegressor 


# Regressione Random Forest

param_grid_rfg = {
    'n_estimators' : [100,200,300],
    'max_depth' : [10,20,None],
    'min_samples_split' : [2,3,5]
}

rfg = RandomForestRegressor(random_state=42)
cv = KFold(n_splits=5, random_state=42, shuffle=True)

grid_rfg = GridSearchCV(estimator=rfg, param_grid=param_grid_rfg, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rfg.fit(X_train_selected, y_train)

best_params = grid_rfg.best_params_
best_estimator = grid_rfg.best_estimator_

rfg_pred_test = best_estimator.predict(X_test_selected)
rfg_pred_train = best_estimator.predict(X_train_selected)

mse_rfg_test = mean_squared_error(y_test, rfg_pred_test)
mse_rfg_train = mean_squared_error(y_train, rfg_pred_train)

print(f"MSE RFG TRAIN: {mse_rfg_train:.2f}")
print(f"MSE RFG TEST: {mse_rfg_test:.2f}")

r2_rfg_test = r2_score(y_test, rfg_pred_test)
r2_rfg_train = r2_score(y_train, rfg_pred_train)

print(f"R2 RFG TRAIN: {r2_rfg_train:.2f}")
print(f"R2 RFG TEST: {r2_rfg_test:.2f}")

print(f"Best Params: {best_params}")
print(f"Best Estimator: {best_estimator}")



MSE RFG TRAIN: 328334071.25
MSE RFG TEST: 2425690022.93
R2 RFG TRAIN: 0.98
R2 RFG TEST: 0.82
Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
Best Estimator: RandomForestRegressor(n_estimators=300, random_state=42)
